# Colabで検証済みの高速版：固定ソース・再開・品質評価
全件解析が実行中ならF4は実行しないでください。既存結果から品質評価する場合はF1→F2→F5→F6。新たな抽出は任意です。

## F1 固定ソース取得・ハッシュ照合・合成テスト
署名対象のソースを変更せず取得します。実牌譜やモデルは実行しません。

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,json,subprocess,urllib.request,hashlib
COMMIT='860605cf0f35f376be9dbd79763775c4300f11b4'
BASE=f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{COMMIT}/research_v2/fast_release_20261126'
HOME=Path('/tmp')/('received_fast_'+COMMIT[:12])
SOURCE=HOME/'snapshot/research_v2'
manifest=json.loads(urllib.request.urlopen(BASE+'/source_manifest.json').read())
for relative,expected in manifest.items():
    p=HOME/'snapshot'/relative;p.parent.mkdir(parents=True,exist_ok=True)
    content=urllib.request.urlopen(BASE+'/snapshot/'+relative).read()
    if hashlib.sha256(content).hexdigest()!=expected: raise ValueError('固定ソース不一致: '+relative)
    if p.exists() and p.read_bytes()!=content: raise ValueError('ローカル固定版が変更されています: '+relative)
    if not p.exists():p.write_bytes(content)
for name in ['quality_supplement.py','test_supplement.py']:
    (HOME/name).write_bytes(urllib.request.urlopen(BASE+'/'+name).read())
DEPS=HOME/'deps';stamp=DEPS/'installed'
if not stamp.exists():
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(DEPS),'-r',str(SOURCE/'features_100/requirements.txt')])
    stamp.write_text(COMMIT)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[DEPS,HOME,SOURCE,SOURCE/'choices',SOURCE/'smoothing'])),PYTHONIOENCODING='utf-8',MPLBACKEND='Agg')
subprocess.check_call([sys.executable,'-m','unittest','discover','-s',str(SOURCE/'features_100/tests'),'-t',str(SOURCE),'-q'],env=ENV)
subprocess.check_call([sys.executable,'-m','unittest','test_supplement','-q'],env=ENV,cwd=HOME)
subprocess.run(['apt-get','install','-y','-qq','fonts-noto-cjk'],check=False,stdout=subprocess.DEVNULL)
print('固定ソース照合完了。実牌譜の検証完了ではありません。')


Mounted at /content/drive
固定ソース照合完了。実牌譜の検証完了ではありません。


## F2 入力・出力・既存全件結果の指定
OUTPUT_ROOTとcache_shardsは受領設定を維持。探索末とその根拠は自分で設定し、F5で両群の窓数を確認してください。過去に閲覧した期間は独立評価には戻せません。

In [2]:
ROOT=Path('/content/drive/MyDrive/卒論')
CONFIG=dict(run_dir=str(ROOT/'research_v2_runs/20261105T071722Z_5da6c9ad'),
    output_root=str(ROOT/'research_v2_features_100_fast'),player='しょぼんｎ',key='shobon',
    zip_override=None,pilot_per_group=3,pilot_ids=None,workers=2,cache_shards=True,
    exploration_end_hour='',exploration_boundary_reason='')
EXTRACTION_DIR = '/content/drive/MyDrive/卒論/research_v2_features_100_fast/20261126T100602Z_994332bc' # 全件解析完了フォルダ（complete.jsonがある日時フォルダ）を設定
MODE='B' # 追加少数検証はA。実行中のBとは同時実行しない。
CFG=HOME/'config.json';RECEIPT=HOME/('preview_'+MODE+'.json')
def save_config():CFG.write_text(json.dumps(CONFIG,ensure_ascii=False),encoding='utf8')
def step(action,extra=()):
    save_config()
    subprocess.check_call([sys.executable,'-m','features_100.cli',action,str(CFG),'--mode',MODE,'--receipt',str(RECEIPT),*extra],env=ENV)
save_config()


In [5]:

# F2.5 探索期間の候補確認

assert EXTRACTION_DIR, "F2でEXTRACTION_DIRを設定してください"
assert (Path(EXTRACTION_DIR) / "complete.json").exists(), \
    "全件抽出の完了ファイルが見つかりません"

script = r"""
import sys
import json
import pandas as pd
from pathlib import Path
from decision_features.pipeline import load_c5
from mahjong_research import tenhou
from mahjong_research.analysis import validate_data
from study import make_blocks

config = json.loads(
    Path(sys.argv[1]).read_text(encoding="utf-8")
)

data, shared, _, _ = load_c5(
    config["run_dir"],
    config["key"],
    config["player"]
)

d, cutoff = validate_data(data, tenhou.FEATURE_NAMES)

blocks = make_blocks(d, cutoff, shared)
rows = []

for width in (10, 20):
    for start in blocks["available"][width]:
        p = d.iloc[start:start + width]
        rows.append({
            "width": width,
            "label": int(p.target_label.iloc[0]),
            "first": p.hour.iloc[0],
            "last": p.hour.iloc[-1]
        })

w = pd.DataFrame(rows)

print("=== 基本情報 ===")
print("全対局数:", len(d))
print("初期除外境界:", cutoff)
print("最終対局:", d.hour.max())

print("\n=== 年別・段位別の10対局窓数 ===")
a = w[w.width == 10].copy()
a["year"] = a["last"].dt.year

print(pd.crosstab(
    a["year"], a["label"]
).rename(columns={0: "低段位", 1: "高段位"}))

print("\n=== 探索期間の境界候補 ===")

candidates = pd.date_range(
    start=cutoff,
    end=d.hour.max(),
    freq="QS"
) - pd.Timedelta(hours=1)

results = []

for end in candidates:
    if not cutoff < end < d.hour.max():
        continue

    row = {"境界": str(end)}

    for width in (10, 20):
        z = w[w.width == width]

        for label, name in [(0, "低"), (1, "高")]:
            row[f"探索{width}_{name}"] = int(
                ((z.label == label) &
                 (z["last"] <= end)).sum()
            )
            row[f"評価{width}_{name}"] = int(
                ((z.label == label) &
                 (z["first"] > end)).sum()
            )

    results.append(row)

r = pd.DataFrame(results)

cols = [c for c in r.columns if c != "境界"]

valid = r[(r[cols] > 0).all(axis=1)]

if len(valid):
    print(valid.to_string(index=False))
else:
    print("全期間・両段位群を確保できる候補なし")
    print(r.to_string(index=False))

print("\n=== 確認完了 ===")
"""

subprocess.check_call(
    [sys.executable, "-u", "-c", script, str(CFG)],
    env=ENV
)


0

In [6]:

# F2.5-b 実行結果の表示

result = subprocess.run(
    [sys.executable, "-u", "-c", script, str(CFG)],
    env=ENV,
    capture_output=True,
    text=True,
    encoding="utf-8"
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

print("終了コード:", result.returncode)

=== 基本情報 ===
全対局数: 11573
初期除外境界: 2012-06-23 03:00:00
最終対局: 2026-05-15 22:00:00

=== 年別・段位別の10対局窓数 ===
label  低段位  高段位
year           
2012    62    0
2013    34    0
2014    16   29
2015     7    9
2016    15    0
2017    36    0
2019     0   10
2020    21    0
2021    44    0
2022    49    0
2023    40    0
2024    48   25
2025    76   28
2026     0   21

=== 探索期間の境界候補 ===
                 境界  探索10_低  評価10_低  探索10_高  評価10_高  探索20_低  評価20_低  探索20_高  評価20_高
2014-07-01 02:00:00     112     336       6     115      56     167       3      54
2014-10-01 02:00:00     112     336      19     102      56     167       8      49
2015-01-01 02:00:00     112     336      29      92      56     167      13      44
2015-04-01 02:00:00     112     336      38      84      56     167      17      41
2015-07-01 02:00:00     113     334      38      84      56     166      17      41
2015-10-01 02:00:00     119     328      38      84      59     163      17      41
2016-01-01 02:00:00     119     328

In [7]:

# F2.6 探索期間の確定

CONFIG["exploration_end_hour"] = "2024-07-01 02:00:00"

CONFIG["exploration_boundary_reason"] = (
    "特徴量の値や分類性能を参照せず、"
    "非重複10対局窓および20対局窓の段位群別件数に基づいて設定。"
    "少数群である高段位群を探索・評価期間でほぼ均等に分割し、"
    "低段位群についても両期間のサンプル数を確保するため。"
)

EXTRACTION_DIR = (
    "/content/drive/MyDrive/卒論/"
    "research_v2_features_100_fast/"
    "20261126T100602Z_994332bc"
)

save_config()

print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
print("抽出結果:", EXTRACTION_DIR)


{
  "run_dir": "/content/drive/MyDrive/卒論/research_v2_runs/20261105T071722Z_5da6c9ad",
  "output_root": "/content/drive/MyDrive/卒論/research_v2_features_100_fast",
  "player": "しょぼんｎ",
  "key": "shobon",
  "zip_override": null,
  "pilot_per_group": 3,
  "pilot_ids": null,
  "workers": 2,
  "cache_shards": true,
  "exploration_end_hour": "2024-07-01 02:00:00",
  "exploration_boundary_reason": "特徴量の値や分類性能を参照せず、非重複10対局窓および20対局窓の段位群別件数に基づいて設定。少数群である高段位群を探索・評価期間でほぼ均等に分割し、低段位群についても両期間のサンプル数を確保するため。"
}
抽出結果: /content/drive/MyDrive/卒論/research_v2_features_100_fast/20261126T100602Z_994332bc


## F3 任意：再開前の署名・件数確認
F4で再開する場合だけ実行。既存キャッシュとソースの整合性は各キャッシュ読込時にも検証します。ロック・一時ファイルを自動削除しません。

In [ ]:
step('preview')


## F4 任意：中断後の再開／少数追加抽出
実行中の全件解析がある場合は実行しない。成功キャッシュを再利用し、失敗を除外して窓を詰めません。残存ロックや署名不一致があればREADMEの手順に従い原因を確認してください。

In [ ]:
RUN_EXTRACTION=False
if RUN_EXTRACTION:
    step('extract')
    EXTRACTION_DIR=json.loads(RECEIPT.with_suffix('.result.json').read_text())['output']
else:print('抽出をスキップ。保存済み全件結果からF5〜F6を実行できます。')


## F5 探索境界の確認（ZIP再解析なし）
探索期間末をF2に設定してから実行。両群の窓数があることは必要条件であり、十分な行動機会を保証しません。境界を性能が良くなるように選ばないでください。

In [9]:

# F5 探索境界の確認

save_config()

if not CONFIG["exploration_end_hour"]:
    raise ValueError("探索期間末を設定してください")

result = subprocess.run(
    [
        sys.executable,
        str(HOME / "quality_supplement.py"),
        "boundary",
        str(CFG)
    ],
    env=ENV,
    capture_output=True,
    text=True,
    encoding="utf-8"
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

print("終了コード:", result.returncode)
result.check_returncode()


       scope  width  label         partition  windows
   available      1      0       exploration     3467
common_games      1      0       exploration     2440
   available      1      0          reserved     1261
common_games      1      0          reserved     1140
   available      1      1       exploration      652
   available      1      1          reserved      622
common_games      1      1          reserved      471
common_games      1      1       exploration      429
   available     10      0       exploration      324
common_games     10      0       exploration      244
   available     20      0       exploration      162
   available     10      0          reserved      124
common_games     20      0       exploration      122
common_games     10      0          reserved      114
   available     10      1          reserved       61
   available     20      0          reserved       61
   available     10      1       exploration       60
common_games     20      0  

## F6 全件抽出の品質評価＋群別相関
既存署名・CSVハッシュ・C5のID/段位/座席/XML対応を検証後、新しい日時フォルダに出力。分母0・unknownは補完しません。モデル学習なし。

In [ ]:

# F6 全件抽出の品質評価＋群別相関

if not EXTRACTION_DIR:
    raise ValueError("全件抽出完了フォルダを指定してください")

save_config()

print("90特徴量の品質評価を開始します")

result = subprocess.run(
    [
        sys.executable,
        "-u",
        str(HOME / "quality_supplement.py"),
        "quality",
        str(CFG),
        "--extraction-dir",
        str(EXTRACTION_DIR)
    ],
    env=ENV,
    capture_output=True,
    text=True,
    encoding="utf-8"
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

print("終了コード:", result.returncode)
result.check_returncode()

print("出力親:", Path(CONFIG["output_root"]) / "quality")


90特徴量の品質評価を開始します


In [ ]:
if not EXTRACTION_DIR:raise ValueError('F2で全件抽出完了フォルダを指定してください')
save_config()
subprocess.check_call([sys.executable,str(HOME/'quality_supplement.py'),'quality',str(CFG),'--extraction-dir',str(EXTRACTION_DIR)],env=ENV)
print('出力:',Path(CONFIG['output_root'])/'quality','の新しい日時フォルダ')


## 確認する成果物
group_statistics.csv、feature_comparison.csv、period_and_exposure_statistics.csv、sampling_noise_reference.csv、feature_correlations.csv、partition_counts.csv、boundary_rationale.json、report_ja.md、figures/。
小分母・局内依存・時間と段位の交絡に注意。高相関や低分散だけで採否を決めません。副露直前情報・Z独立検算・未観測候補の監査は別途残っています。